### 3.9.29. Coordinate Descent

$$
x_i^{k+1}
= \arg\min_{\xi\in\mathbb{R}}
f(x_1^{k+1},\ldots,x_{i-1}^{k+1},\xi,x_{i+1}^k,\ldots,x_n^k).
$$


**Explanation:**

Coordinate descent minimizes the objective along one coordinate direction at a time while holding the other coordinates fixed. It is simple, often easy to implement, and useful when one-dimensional subproblems are cheap. It fits naturally among nonderivative methods because it can be used even when full gradients are not explicitly available, although it can also be applied to differentiable functions. Its convergence behavior is often comparable to steepest descent: reliable under suitable assumptions, but sometimes slow.

The formula updates only coordinate $i$ by minimizing over the scalar variable $\xi$. Coordinates already updated in the sweep use their new values, while later coordinates still use their old values from iteration $k$.

The method is especially natural when the variables have separable or weakly coupled structure. In such cases, updating one coordinate may be inexpensive, and independent coordinate blocks may even be processed in parallel. The main limitation is that coordinate directions may be poorly aligned with the valley of the objective, producing slow progress on strongly coupled problems.

**Intuition:**

For a quadratic cost, minimizing with respect to one coordinate means solving one scalar equation while keeping the remaining coordinates fixed. The figure illustrates the resulting axis-aligned movement through level sets. The code cell performs one full coordinate sweep on a two-variable [positive definite](../../01_Foundations/01_Linear_Algebra/07_Theoretical_Linear_Algebra/17_positive_definite_matrix.ipynb) quadratic.

<img src="../../Figures/030528_berts_fig_1_8_1_coordinate_descent.jpeg" alt="Bertsekas Figure 1.8.1: coordinate descent method" style="width: 60%; height: auto;">


**Numerical Example:**

The coordinate-descent figure shows axis-aligned moves. Let
$$
Q=\begin{bmatrix}4&1\\1&2\end{bmatrix},
\qquad b=\begin{bmatrix}1\\3\end{bmatrix},
\qquad x^0=\begin{bmatrix}0\\0\end{bmatrix}.
$$

For the quadratic $f(x)=\frac{1}{2}x^{\mathsf T}Qx-b^{\mathsf T}x$, the coordinate gradient components are $Q_i x-b_i$. First update coordinate $1$:
$$
r_1=\begin{bmatrix}4&1\end{bmatrix}\begin{bmatrix}0\\0\end{bmatrix}-1=-1,
\qquad
x_1\leftarrow 0-\frac{-1}{4}=0.25.
$$

Now update coordinate $2$ using the new first coordinate:
$$
r_2=\begin{bmatrix}1&2\end{bmatrix}\begin{bmatrix}0.25\\0\end{bmatrix}-3=0.25-3=-2.75,
$$
so
$$
x_2\leftarrow 0-\frac{-2.75}{2}=1.375.
$$

After one coordinate sweep,
$$
x=\begin{bmatrix}0.25\\1.375\end{bmatrix}.
$$


In [1]:
import sympy as sp

x_1, x_2 = sp.symbols("x_1 x_2")
decision_variable = sp.Matrix([x_1, x_2])
Q = sp.Matrix([[4, 1], [1, 2]])
linear_term = sp.Matrix([1, 3])
objective = (decision_variable.T * Q * decision_variable)[0] / 2 - (linear_term.T * decision_variable)[0]
gradient = sp.Matrix([sp.diff(objective, variable) for variable in decision_variable])

current_point = sp.Matrix([0, 0])
for coordinate in range(2):
    partial_derivative = gradient[coordinate].subs({x_1: current_point[0], x_2: current_point[1]})
    current_point[coordinate] = current_point[coordinate] - partial_derivative / Q[coordinate, coordinate]

print("gradient ="); sp.pprint(gradient)
print("current_point ="); sp.pprint(current_point)

gradient =
⎡4⋅x₁ + x₂ - 1⎤
⎢             ⎥
⎣x₁ + 2⋅x₂ - 3⎦
current_point =
⎡1/4 ⎤
⎢    ⎥
⎣11/8⎦


**Equivalent `casadi` implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x", 2)
Q = ca.DM([[4, 1], [1, 2]])
linear_term = ca.DM([1, 3])
objective = 0.5 * ca.bilin(Q, decision_variable, decision_variable) - ca.dot(linear_term, decision_variable)
hessian, gradient_expr = ca.hessian(objective, decision_variable)
gradient_function = ca.Function("grad", [decision_variable], [gradient_expr])
hessian_matrix = ca.Function("hessian_eval", [decision_variable], [hessian])(ca.DM([0, 0]))

current_point = ca.DM([0, 0])
print("initial_point =", current_point)

for coordinate in range(2):
    partial_derivative = gradient_function(current_point)[coordinate]
    coordinate_step = -partial_derivative / hessian_matrix[coordinate, coordinate]
    current_point[coordinate] = current_point[coordinate] + coordinate_step
    print(f"coordinate {coordinate + 1} partial_derivative =", partial_derivative)
    print(f"coordinate {coordinate + 1} coordinate_step =", coordinate_step)
    print(f"coordinate {coordinate + 1} current_point =", current_point)

print("final_point =", current_point)


initial_point = [0, 0]
coordinate 1 partial_derivative = -1
coordinate 1 coordinate_step = 0.25
coordinate 1 current_point = [0.25, 0]
coordinate 2 partial_derivative = -2.75
coordinate 2 coordinate_step = 1.375
coordinate 2 current_point = [0.25, 1.375]
final_point = [0.25, 1.375]


**References:**

[📘 Bertsekas, D. P. (1999). *Nonlinear Programming* (2nd ed.), Chapter 1 "Unconstrained Optimization". Athena Scientific.](https://www.athenasc.com/nonlinbook.html)

---

[⬅️ Previous: Incremental Gradient Methods](./28_incremental_gradient_methods.ipynb) | [Next: Finite Difference Derivative Approximations ➡️](./30_finite_difference_derivative_approximations.ipynb)

---
